In [1]:
import numpy as np 
import pandas as pd
import math
import random
import seaborn as sns

from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import manhattan_distances, euclidean_distances
from sklearn.cluster import KMeans

from Clustering_Functions import *

In [8]:
full_party_list = [
'Alba Party for Independence (API)',
'Borders (Borders)',
'Britannica Party (BP)',
'British National Party (BNP)',
'British Unionists (BU)',
'Christian Peoples Alliance (CPA)',
'Communist Party of Britain (Comm)',
'(Scottish) Conservative and Unionist Party (Con)',
'Cumbernauld Independent Councillors Alliance (CICA)',
'East Dunbartonshire Independent Alliance (EDIA)',
'East Kilbride Alliance (EKA)',
'Freedom Alliance (FA)',
'Glasgow First (Glasgow First)',
'(Scottish) Green (Gr)',
'Independence for Scotland Party (ISP)',
'Independent (Ind)',
'Independent Alliance North Lanarkshire (IANL)',
'Labour (Lab)',
'Labour and Co-operative Party (LabCo)',
'Liberal (Lib)',
'Liberal Democrat (LD)',
'Libertarian (Libtn)',
'Monster Raving Loony (MVR)',
'National Front (NF)',
'No Referendum, Maintain Union, Pro-Brexit (NRMUPB)',
'Orkney Manifesto Group (OMG)',
'Piarate (Pir)',
'RISE (RISE)',
'Rubbish (Rubbish)',
'Scotland Independent Network (ScIN)',
'Scottish Christian (SC)',
'Scottish Eco-Federalist Party (SEFP)',
'Scottish Family Party (SFP)',
'Scottish National Party (SNP)',
'Scottish Senior Citizens (SSC)',
'Scottish Unionist (SU)',
'Social Democratic Party (SDP)',
'Socialist (Soc)',
'Socialist Labour Party (SLP)',
'Solidarity (Sol)',
'Sovereignty (Sov)',
"The Pensioner's Party (TPP)",
'Trade Unionist and Socialist Coalition (TUSC)',
'UK Independence Party (UKIP)',
'Vanguard Party (Van)',
'Volt UK (Volt)',
'West Dunbartonshire Community (WDuns)',
"Women's Equality Party (WEP)",
'Worker Party of Britain (WPB)']

In [6]:
results = pd.DataFrame(columns=['filename','num_cands', 'num_voters', 'num_unique_ballots',
                                'avg_ballot_len','ballot_lengths', 'parties', 'method',
                                'block_size', 'silB','silH','centroids_H', 'centroids_B', 
                                'medoids_H','medoids_B'])

In [9]:
for ward in range(2, 18): # need to instead iterate over all 1000+ elections.
    filename = f"Data/edinburgh17-{ward:02}.blt"
    print(filename)

    # compute num_cands, num_unique_ballots, num_voters, avg_ballot_len
    election, cand_names, location = parse(filename)
    num_cands = max([item for ranking in election.keys() for item in ranking])
    all_ballots = [ballot for ballot in election.keys() if election[ballot]>0]
    num_unique_ballots = len(all_ballots)
    candidates = sorted(list(set([item for ranking in all_ballots for item in ranking])))
    num_cands = len(candidates)
    num_voters = sum([election[ballot] for ballot in all_ballots])
    avg_ballot_len = sum([len(ballot)*election[ballot] for ballot in election.keys()])/sum(election.values())
    
    # compute dictionary of ballot lengths
    ballot_lengths = {n:0 for n in range(num_cands+1)}
    for ballot in all_ballots:
        l = len(ballot)
        ballot_lengths[l] +=1   
    
    # commpute dictionary of parties
    parties = dict()
    for count in range(len(cand_names)):
        party = cand_names[count][2]
        parties[count+1] = party

    for method in ['meanBC', 'meansBA', 'meanH', 'medoBC', 'medoBA', 'medoH', 'slate']:
        for count in range(2 if method in ['meanBC', 'meansBA', 'meanH'] else 1):
            if method == 'meanBC':
                C = kmeans(election, proxy = 'Borda', borda_style='bord')
            elif method == 'meanBA':
                C = kmeans(election, proxy = 'Borda', borda_style='full_points')
            elif method == 'meanH':
                C = kmeans(election, proxy = 'HH')
            elif method == 'medoBC':
                C = kmedoids(election, proxy = 'Borda', borda_style = 'bord', verbose = False)
            elif method == 'medoBA':
                C = kmedoids(election, proxy = 'Borda', borda_style = 'full_points', verbose = False)
            elif method == 'medoH':
                C = kmedoids(election, proxy = 'HH', verbose = False)
            else:
                C = Slate_cluster(election, verbose = False)

            # compute block size
            block_size = sum(C[0].values())/sum(election.values())

            # compute silhouetted scores 
            labels = []
            XB = []
            XH = [] # first build list of ballot proxies with repititions
            for ballot, weight in election.items():
                for _ in range(weight):
                    XB.append(Borda_vector(ballot, num_cands=num_cands))
                    XH.append(HH_proxy(ballot,num_cands=num_cands))
                    label = 0 if ballot in C[0].keys() else 1
                    labels.append(label)
            silB = silhouette_score(XB,labels,metric='manhattan')
            silH = silhouette_score(XH,labels,metric='manhattan')
            
            # compute the centroids and medoids

            medoids_B = dict()
            medoids_H = dict()
            centroids_B = dict()
            centroids_H = dict()
            for cn in range(2): # cn = cluster number
                centroids_B[cn], medoids_B[cn] = Centroid_and_Medoid(C[cn], proxy='Borda')
                centroids_H[cn], medoids_H[cn] = Centroid_and_Medoid(C[cn], proxy='HH')  

            # save it all in the next row of the dataframe.
            row_num = results.shape[0]
            results.loc[row_num] = [filename,num_cands, num_voters, num_unique_ballots,
                        avg_ballot_len,ballot_lengths, parties, method,
                        block_size, silB, silH, 
                        centroids_H, centroids_B, medoids_H, medoids_B]

Data/edinburgh17-02.blt
Data/edinburgh17-03.blt
Data/edinburgh17-04.blt


KeyboardInterrupt: 

In [10]:
results

,filename,num_cands,num_voters,num_unique_ballots,avg_ballot_len,ballot_lengths,parties,method,block_size,silB,silH,centroids_H,centroids_B,medoids_H,medoids_B
0,Data/edinburgh17-02.blt,7,11315,1238,3.239947,"{0: 0, 1: 7, 2: 39, 3: 170, 4: 271, 5: 174, 6:...","{1: 'C', 2: 'LD', 3: 'SNP', 4: 'Lab', 5: 'SNP'...",meanBC,0.493681,0.509067,0.509842,"{0: [0.4016290726817043, 0.47395273899033297, ...","{0: [5.226100966702471, 1.579484425349087, 0.2...","{0: (1, 6), 1: (3, 5, 4)}","{0: (1, 6), 1: (3, 5, 4, 7)}"
1,Data/edinburgh17-02.blt,7,11315,1238,3.239947,"{0: 0, 1: 7, 2: 39, 3: 170, 4: 271, 5: 174, 6:...","{1: 'C', 2: 'LD', 3: 'SNP', 4: 'Lab', 5: 'SNP'...",meanBC,0.493681,0.509067,0.509842,"{0: [0.4016290726817043, 0.47395273899033297, ...","{0: [5.226100966702471, 1.579484425349087, 0.2...","{0: (1, 6), 1: (3, 5, 4)}","{0: (1, 6), 1: (3, 5, 4, 7)}"
2,Data/edinburgh17-02.blt,7,11315,1238,3.239947,"{0: 0, 1: 7, 2: 39, 3: 170, 4: 271, 5: 174, 6:...","{1: 'C', 2: 'LD', 3: 'SNP', 4: 'Lab', 5: 'SNP'...",meansBA,0.375961,0.496186,0.499423,"{0: [-0.1400446638457922, -0.44422896097790315...","{0: [0.33615420780441935, 1.1475082275505406, ...","{0: (3, 5, 7), 1: (1, 6)}","{0: (3, 5, 7), 1: (1, 6, 4)}"
3,Data/edinburgh17-02.blt,7,11315,1238,3.239947,"{0: 0, 1: 7, 2: 39, 3: 170, 4: 271, 5: 174, 6:...","{1: 'C', 2: 'LD', 3: 'SNP', 4: 'Lab', 5: 'SNP'...",meansBA,0.375961,0.496186,0.499423,"{0: [-0.1400446638457922, -0.44422896097790315...","{0: [0.33615420780441935, 1.1475082275505406, ...","{0: (3, 5, 7), 1: (1, 6)}","{0: (3, 5, 7), 1: (1, 6, 4)}"
4,Data/edinburgh17-02.blt,7,11315,1238,3.239947,"{0: 0, 1: 7, 2: 39, 3: 170, 4: 271, 5: 174, 6:...","{1: 'C', 2: 'LD', 3: 'SNP', 4: 'Lab', 5: 'SNP'...",meanH,0.501105,0.509413,0.510448,"{0: [-0.17328042328042328, -0.3463844797178130...","{0: [0.2890652557319224, 1.5901234567901235, 3...","{0: (3, 5, 4), 1: (1, 6)}","{0: (3, 5, 4, 7), 1: (1, 6)}"
5,Data/edinburgh17-02.blt,7,11315,1238,3.239947,"{0: 0, 1: 7, 2: 39, 3: 170, 4: 271, 5: 174, 6:...","{1: 'C', 2: 'LD', 3: 'SNP', 4: 'Lab', 5: 'SNP'...",meanH,0.501105,0.509413,0.510448,"{0: [-0.17328042328042328, -0.3463844797178130...","{0: [0.2890652557319224, 1.5901234567901235, 3...","{0: (3, 5, 4), 1: (1, 6)}","{0: (3, 5, 4, 7), 1: (1, 6)}"
6,Data/edinburgh17-02.blt,7,11315,1238,3.239947,"{0: 0, 1: 7, 2: 39, 3: 170, 4: 271, 5: 174, 6:...","{1: 'C', 2: 'LD', 3: 'SNP', 4: 'Lab', 5: 'SNP'...",meanBC,0.506319,0.509067,0.509842,"{0: [-0.17254320125676384, -0.339238959678827,...","{0: [0.3154128120090766, 1.6041193925641473, 3...","{0: (3, 5, 4), 1: (1, 6)}","{0: (3, 5, 4, 7), 1: (1, 6)}"
7,Data/edinburgh17-02.blt,7,11315,1238,3.239947,"{0: 0, 1: 7, 2: 39, 3: 170, 4: 271, 5: 174, 6:...","{1: 'C', 2: 'LD', 3: 'SNP', 4: 'Lab', 5: 'SNP'...",meanBC,0.493681,0.509067,0.509842,"{0: [0.4016290726817043, 0.47395273899033297, ...","{0: [5.226100966702471, 1.579484425349087, 0.2...","{0: (1, 6), 1: (3, 5, 4)}","{0: (1, 6), 1: (3, 5, 4, 7)}"
8,Data/edinburgh17-02.blt,7,11315,1238,3.239947,"{0: 0, 1: 7, 2: 39, 3: 170, 4: 271, 5: 174, 6:...","{1: 'C', 2: 'LD', 3: 'SNP', 4: 'Lab', 5: 'SNP'...",meansBA,0.375961,0.496186,0.499423,"{0: [-0.1400446638457922, -0.44422896097790315...","{0: [0.33615420780441935, 1.1475082275505406, ...","{0: (3, 5, 7), 1: (1, 6)}","{0: (3, 5, 7), 1: (1, 6, 4)}"
9,Data/edinburgh17-02.blt,7,11315,1238,3.239947,"{0: 0, 1: 7, 2: 39, 3: 170, 4: 271, 5: 174, 6:...","{1: 'C', 2: 'LD', 3: 'SNP', 4: 'Lab', 5: 'SNP'...",meansBA,0.375961,0.496186,0.499423,"{0: [-0.1400446638457922, -0.44422896097790315...","{0: [0.33615420780441935, 1.1475082275505406, ...","{0: (3, 5, 7), 1: (1, 6)}","{0: (3, 5, 7), 1: (1, 6, 4)}"


In [16]:
election, cand_names, location = parse("Data/edinburgh17-02.blt")

In [30]:
filename = "9_scot_elecs/north_ayrshire_2022_north_coast.csv"
election, cand_names, location = parse(filename)